# Re-DocRED (revised) — GAT(개선 2) 학습 / 평가

`re_model_gat.py`(DocREModelGAT)를 `train_revised`로 학습, `dev_revised`로 평가.
**best-checkpoint tracking + early-stopping(patience 5)** 적용 — dev_F1 최고 epoch만 저장.

**실행 전**: `data/docred_io.py`(revised split)와 `Scripts/atlop/train_gat_es.py`를 **dh에 push** 하세요.
(A100 GPU 런타임 권장. 위→아래 순서대로 실행)


## 1) Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) 코드(dh) + revised 데이터(main, LFS) 가져오기

In [ ]:
%cd /content
!apt-get -qq install -y git-lfs > /dev/null 2>&1
!git lfs install --skip-repo
!rm -rf project1                          # 이전 clone 있으면 제거 (안 그러면 clone 스킵→옛 코드)
!git clone -q https://github.com/multiful/Information_Extraction.git project1
%cd /content/project1
!git checkout -q dh                       # 최종 코드 + train_gat_es.py + docred_io(revised split)
# revised 데이터는 main에 LFS로 있음 -> 가져와 실체화
!git checkout origin/main -- docred_data/data/train_revised.json docred_data/data/dev_revised.json
!git lfs pull --include="docred_data/data/train_revised.json,docred_data/data/dev_revised.json"
!pip install -q transformers==4.57.6 accelerate

# docred_io에 revised split 보장 (스테일 clone/미push 대비 런타임 패치)
import re, pathlib
_p = pathlib.Path("data/docred_io.py"); _s = _p.read_text(encoding="utf-8")
if "train_revised" not in _s:
    _p.write_text(re.sub(r'SPLITS\s*=\s*\[[^\]]*\]',
        'SPLITS = ["train_annotated","train_distant","dev","test","train_revised","dev_revised","test_revised"]',
        _s, count=1), encoding="utf-8")
    print("docred_io SPLITS 런타임 패치")

import os
assert os.path.exists("Scripts/atlop/train_gat_es.py"), \
    "train_gat_es.py 없음 -> dh에 push 했는지 확인!"
for f in ["train_revised", "dev_revised"]:
    kb = os.path.getsize(f"docred_data/data/{f}.json") // 1024
    print(f"{f}: {kb} KB", "OK" if kb > 100 else "  <-- LFS 실체화 실패")

## 3) 인코더(bert-base-cased) 로컬 다운로드 (stall 우회)

In [ ]:
import os
os.makedirs("/content/bert-base-cased", exist_ok=True)
%cd /content/bert-base-cased
B  = "https://huggingface.co/google-bert/bert-base-cased/resolve/main"
UA = "Mozilla/5.0"
for f in ["config.json", "vocab.txt", "tokenizer.json", "tokenizer_config.json", "model.safetensors"]:
    print("↓", f, flush=True)
    !wget -c --tries=200 --timeout=30 --waitretry=3 --header="User-Agent: {UA}" -O {f} {B}/{f}
%cd /content/project1
!ls -lh /content/bert-base-cased/model.safetensors   # ~436M 이면 성공
# 계속 403/멈춤이면 B를 미러로: "https://hf-mirror.com/google-bert/bert-base-cased/resolve/main"

## 4) GAT 학습(train_revised) + 평가(dev_revised) — best-ckpt + early-stopping
- `--model gat` : 개선 2 (LCP + Entity Pair Graph + GAT)
- `--patience 5` : dev_F1이 5 epoch 개선 없으면 중단
- `--epochs 30` : 상한 (ES가 알아서 조기 종료). best epoch만 저장됨
- 매 epoch `dev_F1 / Ign_F1` 출력, best 갱신 시 `*best*` 표시


In [ ]:
!python -u -m Scripts.atlop.train_gat_es \
  --model gat \
  --model_name_or_path /content/bert-base-cased \
  --train_split train_revised --dev_split dev_revised --distant_mode none \
  --epochs 30 --patience 5 --eval_batch_size 32 \
  --run_name atlop_gat_revised --save_model

## 5) 결과 Drive 백업 (필수) — `dst`를 본인 경로로

In [ ]:
dst = "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"   # <- 본인 경로
!cp results/atlop_gat_revised.pt "{dst}/"
!cp results/atlop_gat_revised_dev_predictions.json "{dst}/"
print("✅ Drive 저장 완료:", dst)